# Mental Health Text Risk Detector

This notebook trains a model to predict mental health risks based on raw Reddit data.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

## Data Loading
Load data from the raw directory, mapping filenames to labels.

In [ ]:
raw_data_dir = 'Dataset/Original Reddit Data/raw data/'
all_files = glob.glob(os.path.join(raw_data_dir, '**', '*.csv'), recursive=True)
dfs = []
for f in all_files:
    df = pd.read_csv(f)
    if 'selftext' not in df.columns:
        continue
    
    fname = os.path.basename(f).lower()
    if 'dep' in fname:
        label = 'Depression'
    elif 'ani' in fname or 'anx' in fname:
        label = 'Anxiety'
    elif 'lone' in fname:
        label = 'Loneliness'
    elif 'mh' in fname:
        label = 'Mental Health'
    elif 'sw' in fname or 'suicide' in fname:
        label = 'High Risk (SW)'
    else:
        label = 'Other'
        
    df['Label'] = label
    # Sample up to 2000 rows per file to avoid memory issues
    df_sampled = df.dropna(subset=['selftext']).sample(min(2000, len(df.dropna(subset=['selftext']))), random_state=42)
    dfs.append(df_sampled[['selftext', 'Label']])

train_df = pd.concat(dfs, ignore_index=True)
print("Training data size:", len(train_df))
print(train_df['Label'].value_counts())

## Preprocessing & Feature Extraction
Clean the text and set up the model pipeline.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

train_df['cleaned_text'] = train_df['selftext'].apply(clean_text)

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

pipeline.fit(train_df['cleaned_text'], train_df['Label'])
print("Training complete.")

## Validation
Validate on the labelled data.

In [ ]:
test_file = 'Dataset/Original Reddit Data/Labelled Data/LD DA 1.csv'
test_df = pd.read_csv(test_file)
test_df['cleaned_text'] = test_df['selftext'].apply(clean_text)
predictions = pipeline.predict(test_df['cleaned_text'])
print("Sample predictions on labeled data:")
print(pd.Series(predictions).value_counts())

## Save Model
Save the model so Streamlit can use it.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/model.joblib')
print("Model saved to models/model.joblib")